# Ejercicio: asistente de documentos con NVIDIA NIM y Gradio

**Tiempo:** 40–45 minutos  
**Grupos:** 3–4 personas

## Contexto

En los notebooks anteriores construimos chatbots que responden preguntas
generales. En este ejercicio construiremos un asistente que responda preguntas
sobre un documento específico: **“Attention Is All You Need”**
(Vaswani et al., 2017).

Inyectar un documento completo en el contexto permite construir un primer
prototipo de *document question answering*. Esto ayuda a entender la motivación
de **RAG**, aunque todavía no es RAG: aquí no existe recuperación de fragmentos.

## Objetivo

Construir una aplicación Gradio donde el usuario pueda:

1. Cargar o leer el PDF del paper.
2. Hacer preguntas sobre su contenido.
3. Obtener respuestas basadas exclusivamente en el documento.

## Lo que van a aprender

- Extracción de texto de PDF con `pypdf`.
- Uso de contexto externo en un `system` prompt.
- Consumo de NVIDIA NIM mediante una API compatible con OpenAI.
- Streaming e historial con Gradio.
- Limitaciones de este enfoque y motivación de RAG.

## Paso 0: instalación y configuración

### 0.1 Instala las dependencias

- `pypdf`: extracción del texto del PDF.
- `gradio`: interfaz web.
- `openai`: cliente compatible con NVIDIA NIM.
- `python-dotenv`: lectura de variables de entorno.

```bash
pip install -q --upgrade pypdf gradio openai python-dotenv
```

In [1]:
!pip install -q --upgrade pypdf gradio openai python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.7/402.7 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 17.7 MB/s eta 0:00:00


### 0.2 Descarga el paper

Descarga el PDF abierto desde:

```text
https://arxiv.org/pdf/1706.03762
```

Guárdalo junto al notebook como `attention_is_all_you_need.pdf`.

### 0.3 Configura tus credenciales

Crea un archivo `.env` con tu API key de NVIDIA:

```text
NVIDIA_API_KEY="tu_key_aqui"
```

Puedes crearla en [NVIDIA Build](https://build.nvidia.com/settings/api-keys).

## Paso 1: extracción de texto del PDF

Lee el PDF y extrae su contenido como texto plano.

**Instrucciones:**

1. Importa `PdfReader` desde `pypdf`.
2. Crea `extract_text_from_pdf(pdf_path)`.
3. Abre el PDF e itera sobre `reader.pages`.
4. Concatena el resultado de `page.extract_text()`; contempla páginas sin texto.
5. Retorna el texto completo.
6. Prueba la función e imprime los primeros 500 caracteres.

> **Pista:** usa `page.extract_text() or ""` para evitar valores `None`.

In [3]:
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    full_text = []
    for page in reader.pages:

        text = page.extract_text() or ""
        full_text.append(text)
    return "\n".join(full_text)

pdf_path = "attention_is_all_you_need.pdf"
document_text = extract_text_from_pdf(pdf_path)

print(f"Caracteres extraídos: {len(document_text)}")
print("--- Primeros 500 caracteres ---")
print(document_text[:500])

Caracteres extraídos: 39599
--- Primeros 500 caracteres ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


## Paso 2: inicialización del cliente de NVIDIA NIM

**Instrucciones:**

1. Importa `os`, `load_dotenv` y `OpenAI`.
2. Carga las variables con `load_dotenv()`.
3. Verifica que `NVIDIA_API_KEY` esté disponible sin imprimirla.
4. Inicializa `OpenAI` con:
   - `base_url="https://integrate.api.nvidia.com/v1"`
   - `api_key=os.getenv("NVIDIA_API_KEY")`
5. Define:

```python
MODELO = "nvidia/nemotron-3-super-120b-a12b"
```

In [7]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("NVIDIA_API_KEY")
if not api_key:
    raise ValueError("NVIDIA_API_KEY no encontrada. Por favor, verifica tu archivo .env.")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=api_key
)

MODELO = "nvidia/nemotron-3-super-120b-a12b"
print("Cliente de NVIDIA NIM inicializado correctamente.")

Cliente de NVIDIA NIM inicializado correctamente.


## Paso 3: función de chat con el documento

Construye un `system` prompt que incluya el texto del paper y obligue al modelo
a fundamentar sus respuestas en él.

**Instrucciones:**

1. Crea `build_system_prompt(document_text)` para:
   - definir al asistente como experto en el paper;
   - incluir el documento entre delimitadores claros;
   - exigir respuestas basadas únicamente en el documento;
   - indicar que responda “No encuentro esa información en el documento” cuando corresponda.

2. Crea `history_to_messages(history)` para convertir el historial de Gradio a
   mensajes con roles `user` y `assistant`.

3. Crea `chat_con_documento(message, history, document_text)` para:
   - iniciar `messages` con el `system` prompt;
   - agregar el historial y el mensaje actual;
   - llamar `client.chat.completions.create(..., stream=True)`;
   - acumular `chunk.choices[0].delta.content`;
   - usar `yield` para entregar la respuesta progresivamente.

> **Pista:** utiliza `temperature=1.0` y `top_p=0.95`, valores recomendados para
> Nemotron 3 Super.

In [12]:
def build_system_prompt(document_text: str) -> str:
    return f"""Eres un asistente experto en el artículo científico "Attention Is All You Need".
Responde a las preguntas del usuario basándote únicamente en el contenido del siguiente documento.

El documento contiene delimitadores de página como "--- [PÁGINA X] ---".
Es OBLIGATORIO que al responder indiques siempre de qué página o páginas obtuviste la información utilizando el formato "[Pág. X]".

=== INICIO DEL DOCUMENTO ===
{document_text}
=== FIN DEL DOCUMENTO ===

Instrucciones estrictas:
- Tus respuestas deben fundamentarse exclusivamente en la información presente en el documento proporcionado.
- Incluye siempre el número de página de donde extrajiste la respuesta (por ejemplo: "Según la [Pág. 3]...").
- Si la información no se encuentra en el documento, responde únicamente: "No encuentro esa información en el documento."
- No utilices conocimiento externo previo bajo ninguna circunstancia.
- Responde siempre en español.
"""

def history_to_messages(history):
    messages = []
    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        if assistant_msg:
            messages.append({"role": "assistant", "content": assistant_msg})
    return messages

def chat_con_documento(message, history, document_text):
    system_prompt = build_system_prompt(document_text)

    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(history_to_messages(history))
    messages.append({"role": "user", "content": message})

    response = client.chat.completions.create(
        model=MODELO,
        messages=messages,
        temperature=1.0,
        top_p=0.95,
        stream=True
    )

    partial_response = ""
    for chunk in response:
        delta = chunk.choices[0].delta.content or ""
        partial_response += delta
        yield partial_response

## Paso 4: interfaz Gradio

Construye la interfaz con `gr.ChatInterface`.

**Instrucciones:**

1. Extrae el texto del PDF con la función del paso 1.
2. Muestra cuántas páginas y caracteres se extrajeron.
3. Configura `gr.ChatInterface` con:
   - `fn=chat_con_documento`;
   - un título y una descripción;
   - un `gr.Textbox` oculto dentro de `additional_inputs` para enviar el documento;
   - al menos tres preguntas de ejemplo.

```python
gr.Textbox(value=document_text, visible=False)
```

4. Lanza la interfaz:

```python
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)
```

In [13]:
import gradio as gr
from pypdf import PdfReader

def extract_text_with_page_markers(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    marked_text_parts = []
    for i, page in enumerate(reader.pages):
        page_num = i + 1
        text = page.extract_text() or ""
        # オプションC: ページ番号のデリミタを挿入
        marked_text_parts.append(f"\n--- [PÁGINA {page_num}] ---\n{text}")
    return "\n".join(marked_text_parts)

pdf_path = "attention_is_all_you_need.pdf"
document_text = extract_text_with_page_markers(pdf_path)
reader = PdfReader(pdf_path)

# オプションA: トークン数と文字数の推定計算と表示
num_chars = len(document_text)
# 一般的に英語では「1トークン ≒ 4文字（スペース含む）」または「1単語 ≒ 1.3トークン」として推測します
estimated_tokens = int(num_chars / 4)

print(f"--- [Análisis de Contexto (Opción A)] ---")
print(f"Número total de páginas: {len(reader.pages)}")
print(f"Número total de caracteres: {num_chars}")
print(f"Número estimado de tokens (aprox. 4 caracteres por token): {estimated_tokens}")
print("Nota: Los caracteres miden el almacenamiento de texto plano, mientras que los tokens representan las unidades semánticas (subpalabras) que el LLM procesa.")
print("------------------------------------------")

examples = [
    ["¿Cuál es la arquitectura principal propuesta en el paper?", document_text],
    ["¿Qué es el mecanismo de atención?", document_text],
    ["¿Cuántas capas tiene el encoder del modelo base?", document_text],
    ["¿Quiénes son los autores del paper?", document_text],
    ["¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?", document_text],
    ["¿Qué es GPT-4?", document_text]
]

# Gradio UI を構築
demo = gr.ChatInterface(
    fn=chat_con_documento,
    title="Asistente con Evidencia de Páginas y Medición de Contexto",
    description=f"""Haz preguntas sobre el artículo 'Attention Is All You Need'.

💡 **Información de Contexto (Opción A):**
- **Caracteres totales:** {num_chars}
- **Tokens estimados:** ~{estimated_tokens} tokens
- *Nota: Los caracteres y tokens difieren porque los modelos de lenguaje agrupan caracteres comunes en 'tokens' para procesar el lenguaje de manera más eficiente (ej: palabras comunes como 'the' o raíces de palabras).*

📖 **Evidencias (Opción C):** El modelo está obligado a citar el número de página `[Pág. X]` de donde obtuvo cada respuesta.""",
    additional_inputs=[
        gr.Textbox(value=document_text, visible=False)
    ],
    examples=examples
)

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", show_error=True)

--- [Análisis de Contexto (Opción A)] ---
Número total de páginas: 15
Número total de caracteres: 39905
Número estimado de tokens (aprox. 4 caracteres por token): 9976
Nota: Los caracteres miden el almacenamiento de texto plano, mientras que los tokens representan las unidades semánticas (subpalabras) que el LLM procesa.
------------------------------------------
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0334db0fa16dab752f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Paso 5: Prueba y reflexión

Una vez que la interfaz esté funcionando, prueba estas preguntas:

1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*
2. *"¿Qué es el mecanismo de atención?"*
3. *"¿Cuántas capas tiene el encoder del modelo base?"*
4. *"¿Quiénes son los autores del paper?"*
5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*

Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*

Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.
¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?


## Paso 6 (adicional): mejora el sistema

Si terminan antes, implementen una mejora:

**A) Medición del contexto**  
Reporta la cantidad de caracteres o estima los tokens del documento. Explica
por qué caracteres y tokens no son equivalentes.

**B) Subida dinámica de PDF**  
Agrega `gr.File` para cargar cualquier PDF y extrae el texto dentro del flujo de
la aplicación.

**C) Evidencia y páginas**  
Conserva separadores de página al extraer el PDF y exige que cada respuesta cite
la página que contiene la evidencia.

**D) Primer paso hacia RAG**  
Divide el documento en fragmentos y selecciona los más relevantes mediante una
búsqueda simple por palabras antes de llamar al modelo.

## Reflexión final

Discutan en grupo:

1. ¿Cuál es la limitación de enviar siempre el documento completo?
- Supera el límite de tokens de la ventana de contexto, incrementa los costos y la latencia de la API, y reduce la precisión del modelo al localizar información específica.
2. ¿Por qué este ejercicio todavía no implementa RAG?
- Porque no realiza una búsqueda o recuperación (retrieval) previa de los fragmentos relevantes, sino que inyecta todo el documento directamente en el prompt.
3. ¿Cómo comprobarían que la respuesta proviene del documento y no del
   conocimiento previo del modelo?
- Exigiendo citas con número de página [Pág. X] y realizando preguntas de control sobre temas no incluidos en el texto (como GPT-4) para comprobar si responde que la información no se encuentra en el documento.
4. ¿Qué riesgos aparecen al insertar directamente documentos de usuarios en el
   prompt?
- Vulnerabilidad a ataques de prompt injection indirecto (instrucciones maliciosas dentro del PDF), riesgos de privacidad por filtración de datos sensibles y conflicto con las instrucciones del sistema.
## Recursos

- [pypdf](https://pypdf.readthedocs.io)
- [Gradio](https://www.gradio.app/docs)
- [NVIDIA Build](https://build.nvidia.com)
- [Nemotron 3 Super](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b)
- [Paper original](https://arxiv.org/abs/1706.03762)